# Regressão Logística I
## Tarefa II

Vamos trabalhar com a mesma base do exercício anterior, mas vamos aprofundar um pouco mais a nossa regressão.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve
from sklearn import metrics
from scipy.stats import ks_2samp

import statsmodels.formula.api as smf

In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'

df = pd.read_csv(url, 
                 names=['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
                        'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num'])
df['flag_doente'] = (df['num']!=0).astype('int64')
df.head()

A descrição das variáveis está recortada abaixo:
- age: idade do paciente em anos
- sex: sexo (1 = male; 0 = female)  
- cp: tipo de dor no peito
  - 1: angina típica
  - 2: angina atípica
  - 3: dor não-angina
  - 4: assintomático
- trestbps: pressão sanguínea em repouso (em mm Hg na admissão ao hospital
- chol: colesterol sérico em mg/dl
- fbs: (açúcar no sangue em jejum > 120 mg/dl) (1 = True; 0 = False)
- restecg: resultados eletrocardiográficos em repouso
  - 0: normal
  - 1: tendo anormalidade da onda ST-T (Inversões de onda T e / ou ST com elevação ou depressão de > 0.05 mV)
  - 2: mostrando hipertrofia ventricular esquerda provável ou definitiva pelos critérios de Estes
- thalach: frequência cardíaca máxima alcançada
- exang: angina induzida por exercício(1 = sim; 0 = não)
- oldpeak = Depressão de ST induzida por exercício em relação ao repouso
- slope: Depressão de ST induzida por exercício em relação ao repouso
  - 1: inclinação ascendente
  - 2: estável
  - 3: inclinação descendente
- ca: número de vasos principais (0-3) coloridos por fluorosopia
- thal: 3 = normal; 6 = defeito corrigido; 7 = defeito reversível
- num: diagnóstico de doença cardíaga (status de doença angiográfica)

In [ ]:
df0 = df.copy()

In [ ]:
df.info()

In [ ]:
(df.isna().any().any())

1. Considere o script que monta a análise bivariada que você fez na tarefa anterior. Transforme esse script em uma função, que deve:
- Ter como parâmetros de entrada:
    - Um *dataframe* contendo os dados a serem avaliados
    - Um *string* contendo o nome da variável resposta
    - Um *string* contendo o nome da variável explicativa
- E deve retornar um *dataframe* com os dados da bivariada. 
**Monte** a mesma bivariada pelo menos três variáveis qualitativas do *data-frame*. Qual delas parece discriminar mais o risco?

In [ ]:
def analise_bivariada(dataframe, variavel_explicativa, variavel_de_interesse):
    _tab = (
        pd.crosstab(
            dataframe[variavel_explicativa],
            dataframe[variavel_de_interesse],
            margins=True,
        )
        .rename(columns={"All": "Total"}, index={"All": "Total"})
        .assign(Probabilidade=lambda x: round(x[1] / x["Total"], 4) * 100)
        .assign(Odds=lambda x: x[1] / x[0])
        .assign(Odd_ratio_vs_total=lambda x: x["Odds"] / x.loc["Total", "Odds"])
        .reindex(columns=[0, 1, 'Probabilidade',"Odds", "Odd_ratio_vs_total", "Total"])
    )


    return _tab

In [ ]:
tab0 = analise_bivariada(df,'sex', 'flag_doente')
tab0

In [ ]:
tab1 = analise_bivariada(df,'exang', 'flag_doente')
tab1

In [ ]:
tab2 = analise_bivariada(df,'restecg', 'flag_doente')
tab2

Resposta: A variável de angina induzida por exercício é a que apresenta a maior chance de evento para o valor 1.0, sendo 3 vezes maior para indicar qualquer doença cardiaca

2. Monte uma função semelhante para categorizar variáveis quantitativas contínuas (com muitas categorias) como ```age```.  
    Além dos mesmos parâmetros da função anterior, defina mais um parâmetro como número de categorias que você deseja quebrar. Defina um valor '*default*' de 5 grupos para este parâmetro.  

In [ ]:
def analise_bivariada_continua(dataframe, variavel_explicativa, variavel_de_interesse, quantidade_cat =5):
    
    dataframe[f'cat_{variavel_explicativa}'] = pd.qcut(dataframe[variavel_explicativa], quantidade_cat)
    
    _tab = (
        pd.crosstab(
            dataframe[f'cat_{variavel_explicativa}'],
            dataframe[variavel_de_interesse],
            margins=True,
        )
        .rename(columns={"All": "Total"}, index={"All": "Total"})
        .assign(Probabilidade=lambda x: round(x[1] / x["Total"], 4) * 100)
        .assign(Odds=lambda x: x[1] / x[0])
        .assign(Odd_ratio_vs_total=lambda x: x["Odds"] / x.loc["Total", "Odds"])
        .reindex(columns=[0, 1, 'Probabilidade',"Odds", "Odd_ratio_vs_total", "Total"])
    )


    return _tab

In [ ]:
analise_bivariada_continua(df,'age','flag_doente')

3. Construa um modelo de regressão logística com as variáveis qualitativas: ```sex + cp +  trestbps``` e com a variável quantitativa ```age```.

**Interprete os parâmetros.**

In [ ]:
reglog1 = smf.logit('flag_doente ~ C(sex) + C(cp) + trestbps + age', data=df).fit()

In [ ]:
reglog1.summary()

4. Avalie o seu modelo quanto a **calibragem**:
- Calcule a probabilidade de evento predita segundo o seu modelo
- Categorize essa probabilidade em G=5 grupos
- Calcule a probabilidade de evento predita média por grupo
- Calcule a taxa de eventos (média da variável indicadora de eventos) por grupo
- Compare graficamente o valor esperado versus observado para a taxa de maus por grupo

In [ ]:
df['preditos'] = reglog1.predict(df)
df.head()

In [ ]:
analise_bivariada_continua(df, 'preditos', 'flag_doente')
df.head()

In [ ]:
group_reg = df.groupby("cat_preditos", observed=False)
qualid = (
    group_reg[["flag_doente"]]
    .count()
    .rename(columns={"flag_doente": "contagem"})
    .assign(média_preditos=group_reg[["preditos"]].mean())
    .assign(média_doentes=group_reg["flag_doente"].mean())
)
qualid

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)

ax = qualid['média_doentes'].plot(label='%Predito')
ax = qualid['média_preditos'].plot(label='%Observado')

ticks = ax.set_xticks([0, 1, 2, 3, 4])
labels = ax.set_xticklabels([1, 2, 3, 4, 5])
ax.legend(loc="lower right")
ax.set_ylabel('Probabilidade de evento')
ax.set_xlabel('Grupo')

5. Avalie o seu modelo quanto a discriminação calculando acurácia, GINI e KS.

In [ ]:
fpr, tpr, thresholds = metrics.roc_curve(df['flag_doente'], df['preditos'])

plt.figure()
lw = 2

fpr, tpr, thresholds = metrics.roc_curve(df['flag_doente'], df['preditos'])
auc_ = metrics.auc(fpr, tpr)
plt.plot(fpr, tpr, color='darkorange',
         lw=lw, label='ROC curve (area = %0.2f)' % auc_)
plt.plot([0, 1], [0, 1], color='navy', lw=lw, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic example')
plt.legend(loc="lower right")
plt.show()

In [ ]:
acc = metrics.accuracy_score(df['flag_doente'], df['preditos']>.5)
print('Acurácia: {0:.2f}%'.format(acc*100))

#AUC
fpr, tpr, thresholds = metrics.roc_curve(df['flag_doente'], df['preditos'])
auc_ = metrics.auc(fpr, tpr)
#Gini
gini = 2*auc_ -1
#KS
ks = ks_2samp(df.loc[df['flag_doente'] == 1, 'preditos'], df.loc[df['flag_doente'] != 1, 'preditos']).statistic

print('KS: {0:.2f}% \nAUC: {1:.2f}% \nGINI: {2:.2f}%'
      .format(ks*100, auc_*100, gini*100))

6. tente melhorar o modelo obtido, por exemplo inserindo ou removendo variáveis.  
    Avalie as características do seu modelo (calibragem e acurácia).

In [ ]:
reglog2 = smf.logit('flag_doente ~ C(sex) + C(cp) + np.power(trestbps, 2) + np.power(age, 2)', data=df).fit()

df1 = df.drop(columns=['preditos'], errors='ignore').copy()
df1['preditos'] = reglog2.predict(df)

acc = metrics.accuracy_score(df1['flag_doente'], df1['preditos']>.5)
print('Acurácia: {0:.2f}%'.format(acc*100))

#AUC
fpr, tpr, thresholds = metrics.roc_curve(df1['flag_doente'], df1['preditos'])
auc_ = metrics.auc(fpr, tpr)
#Gini
gini = 2*auc_ -1
#KS
ks = ks_2samp(df1.loc[df1['flag_doente'] == 1, 'preditos'], df1.loc[df1['flag_doente'] != 1, 'preditos']).statistic

print('KS: {0:.2f}% \nAUC: {1:.2f}% \nGINI: {2:.2f}%'
      .format(ks*100, auc_*100, gini*100))

In [ ]:
reglog3 = smf.logit('flag_doente ~ C(sex) + C(cp) + np.log(trestbps + .1) + np.log(age + .1)', data=df).fit()

df2 = df.drop(columns=['preditos'], errors='ignore').copy()
df2['preditos'] = reglog3.predict(df)

acc = metrics.accuracy_score(df2['flag_doente'], df2['preditos']>.5)
print('Acurácia: {0:.2f}%'.format(acc*100))

#AUC
fpr, tpr, thresholds = metrics.roc_curve(df2['flag_doente'], df2['preditos'])
auc_ = metrics.auc(fpr, tpr)
#Gini
gini = 2*auc_ -1
#KS
ks = ks_2samp(df2.loc[df2['flag_doente'] == 1, 'preditos'], df2.loc[df2['flag_doente'] != 1, 'preditos']).statistic

print('KS: {0:.2f}% \nAUC: {1:.2f}% \nGINI: {2:.2f}%'
      .format(ks*100, auc_*100, gini*100))

In [ ]:
reglog4 = smf.logit('flag_doente ~ C(sex) + C(cp) + np.exp(trestbps) + np.exp(age)', data=df).fit()

df3 = df.drop(columns=['preditos'], errors='ignore').copy()
df3['preditos'] = reglog4.predict(df)

acc = metrics.accuracy_score(df3['flag_doente'], df3['preditos']>.5)
print('Acurácia: {0:.2f}%'.format(acc*100))

#AUC
fpr, tpr, thresholds = metrics.roc_curve(df3['flag_doente'], df3['preditos'])
auc_ = metrics.auc(fpr, tpr)
#Gini
gini = 2*auc_ -1
#KS
ks = ks_2samp(df3.loc[df3['flag_doente'] == 1, 'preditos'], df3.loc[df3['flag_doente'] != 1, 'preditos']).statistic

print('KS: {0:.2f}% \nAUC: {1:.2f}% \nGINI: {2:.2f}%'
      .format(ks*100, auc_*100, gini*100))

In [ ]:
reglog0 = smf.logit(
    "flag_doente ~ C(sex) + C(cp) +trestbps + age + chol + fbs + restecg + thalach+ exang + oldpeak + slope + ca + thal + num",
    data=df,
).fit()

importancia = abs(reglog0.params).sort_values(ascending=False)
importancia

In [ ]:
reglog5 = smf.logit('flag_doente ~ C(sex) + C(cp) + C(thal) + ca + age', data=df).fit()

df4 = df.drop(columns=['preditos'], errors='ignore').copy()
df4['preditos'] = reglog5.predict(df)

acc = metrics.accuracy_score(df4['flag_doente'], df4['preditos']>.5)
print('Acurácia: {0:.2f}%'.format(acc*100))

#AUC
fpr, tpr, thresholds = metrics.roc_curve(df4['flag_doente'], df4['preditos'])
auc_ = metrics.auc(fpr, tpr)
#Gini
gini = 2*auc_ -1
#KS
ks = ks_2samp(df4.loc[df3['flag_doente'] == 1, 'preditos'], df4.loc[df4['flag_doente'] != 1, 'preditos']).statistic

print('KS: {0:.2f}% \nAUC: {1:.2f}% \nGINI: {2:.2f}%'
      .format(ks*100, auc_*100, gini*100))

Resposta: O último modelo com alterações nas variáveis utilizadas na construção da regressão apresentou melhores resultados que seus predecessores, que estavam se limitando ao tratamento das variáveis com transformações polinomiais e logarítmicas 